In [1]:
from __future__ import annotations

import os
from getpass import getpass
from typing import Any

import httpx
import weaviate
import weaviate.classes as wvc

from weaviate.classes.init import Auth
from weaviate.classes.query import Filter, MetadataQuery

In [2]:
if "WEAVIATE_URL" not in os.environ:
    os.environ["WEAVIATE_URL"] = getpass("WEAVIATE_URL: ")

if "WEAVIATE_API_KEY" not in os.environ:
    os.environ["WEAVIATE_API_KEY"] = getpass("WEAVIATE_API_KEY: ")

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass("GOOGLE_API_KEY: ")

In [3]:
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.environ["WEAVIATE_URL"],
    auth_credentials=Auth.api_key(os.environ["WEAVIATE_API_KEY"]),
    headers={
        "X-Goog-Studio-Api-Key": os.environ["GOOGLE_API_KEY"],
    },
)

print("Client ready:", client.is_ready())

Client ready: True


In [4]:
GOOGLE_EMBEDDING_MODEL = "gemini-embedding-001"
GOOGLE_GENERATIVE_MODEL = "gemini-2.5-flash"

print("Embedding model:", GOOGLE_EMBEDDING_MODEL)
print("Generative model:", GOOGLE_GENERATIVE_MODEL)

Embedding model: gemini-embedding-001
Generative model: gemini-2.5-flash


In [5]:
destination_names = [
    "Barcelona",
    "Rome",
    "Athens",
    "Lisbon",
    "Dubrovnik",
    "Prague",
    "Malaga",
    "Nice",
]

In [6]:
def get_json(
    url: str,
    *,
    params: dict[str, Any] | None = None,
) -> dict[str, Any]:
    headers = {
        "User-Agent": "PythonPracticeNotebook/1.0 educational project",
        "Accept": "application/json",
    }

    with httpx.Client(
        timeout=30.0,
        follow_redirects=True,
        headers=headers,
    ) as http_client:
        response = http_client.get(url, params=params)
        response.raise_for_status()
        return response.json()

In [7]:
def fetch_geocoding(city_name: str) -> dict[str, Any]:
    url = "https://geocoding-api.open-meteo.com/v1/search"

    data = get_json(
        url,
        params={
            "name": city_name,
            "count": 1,
            "language": "en",
            "format": "json",
        },
    )

    results = data.get("results", [])

    if not results:
        raise RuntimeError(f"No geocoding result for city: {city_name}")

    return results[0]

In [8]:
def fetch_weather_forecast(
    *,
    latitude: float,
    longitude: float,
    forecast_days: int = 7,
) -> dict[str, Any]:
    url = "https://api.open-meteo.com/v1/forecast"

    return get_json(
        url,
        params={
            "latitude": latitude,
            "longitude": longitude,
            "daily": [
                "weather_code",
                "temperature_2m_max",
                "temperature_2m_min",
                "precipitation_probability_max",
                "wind_speed_10m_max",
            ],
            "timezone": "auto",
            "forecast_days": forecast_days,
        },
    )

In [9]:
def fetch_country_info(country_name: str) -> dict[str, Any]:
    url = f"https://restcountries.com/v3.1/name/{country_name}"

    try:
        data = get_json(
            url,
            params={
                "fullText": "true",
            },
        )
    except Exception:
        data = get_json(url)

    if not data:
        raise RuntimeError(f"No country info for: {country_name}")

    return data[0]

In [10]:
WEATHER_CODE_MAP = {
    0: "clear sky",
    1: "mainly clear",
    2: "partly cloudy",
    3: "overcast",
    45: "fog",
    48: "depositing rime fog",
    51: "light drizzle",
    53: "moderate drizzle",
    55: "dense drizzle",
    61: "slight rain",
    63: "moderate rain",
    65: "heavy rain",
    71: "slight snow fall",
    73: "moderate snow fall",
    75: "heavy snow fall",
    80: "slight rain showers",
    81: "moderate rain showers",
    82: "violent rain showers",
    95: "thunderstorm",
}


def describe_weather_code(code: int) -> str:
    return WEATHER_CODE_MAP.get(code, f"unknown weather code {code}")

In [11]:
def normalize_geocoding(city_name: str, geocoding_data: dict[str, Any]) -> dict[str, Any]:
    return {
        "city": geocoding_data.get("name", city_name),
        "country": geocoding_data.get("country", "unknown"),
        "country_code": geocoding_data.get("country_code", "unknown"),
        "admin1": geocoding_data.get("admin1", "unknown"),
        "latitude": float(geocoding_data["latitude"]),
        "longitude": float(geocoding_data["longitude"]),
        "timezone": geocoding_data.get("timezone", "unknown"),
        "population": int(geocoding_data.get("population", 0)),
    }

In [12]:
def normalize_weather_forecast(weather_data: dict[str, Any]) -> dict[str, Any]:
    daily = weather_data["daily"]

    weather_codes = daily["weather_code"]
    max_temperatures = daily["temperature_2m_max"]
    min_temperatures = daily["temperature_2m_min"]
    rain_probabilities = daily["precipitation_probability_max"]
    wind_speeds = daily["wind_speed_10m_max"]

    avg_max_temp = sum(max_temperatures) / len(max_temperatures)
    avg_min_temp = sum(min_temperatures) / len(min_temperatures)
    avg_rain_probability = sum(rain_probabilities) / len(rain_probabilities)
    max_wind_speed = max(wind_speeds)

    most_common_weather_code = max(
        set(weather_codes),
        key=weather_codes.count,
    )

    weather_description = describe_weather_code(int(most_common_weather_code))

    if avg_rain_probability >= 60:
        weather_label = "rain risk"
    elif max_wind_speed >= 40:
        weather_label = "windy"
    elif avg_max_temp >= 28:
        weather_label = "hot"
    elif avg_max_temp >= 18:
        weather_label = "pleasant"
    elif avg_max_temp >= 10:
        weather_label = "mild"
    else:
        weather_label = "cold"

    weather_summary = (
        f"Average max temperature: {avg_max_temp:.1f}°C. "
        f"Average min temperature: {avg_min_temp:.1f}°C. "
        f"Average rain probability: {avg_rain_probability:.1f}%. "
        f"Maximum wind speed: {max_wind_speed:.1f} km/h. "
        f"Typical weather: {weather_description}. "
        f"Weather label: {weather_label}."
    )

    return {
        "avg_max_temp_c": round(avg_max_temp, 2),
        "avg_min_temp_c": round(avg_min_temp, 2),
        "avg_rain_probability": round(avg_rain_probability, 2),
        "max_wind_speed_kmh": round(max_wind_speed, 2),
        "weather_description": weather_description,
        "weather_label": weather_label,
        "weather_summary": weather_summary,
    }

In [13]:
def normalize_country_info(country_data: dict[str, Any]) -> dict[str, Any]:
    name = country_data["name"]["common"]
    region = country_data.get("region", "unknown")
    subregion = country_data.get("subregion", "unknown")
    population = int(country_data.get("population", 0))

    capital_values = country_data.get("capital", [])
    capital = capital_values[0] if capital_values else "unknown"

    currencies = country_data.get("currencies", {})
    currency_codes = list(currencies.keys())
    currency = currency_codes[0] if currency_codes else "unknown"

    languages = country_data.get("languages", {})
    language_values = list(languages.values())
    language = language_values[0] if language_values else "unknown"

    country_summary = (
        f"{name} is located in {region}, {subregion}. "
        f"Capital: {capital}. "
        f"Population: {population}. "
        f"Currency: {currency}. "
        f"Main language: {language}."
    )

    return {
        "country_region": region,
        "country_subregion": subregion,
        "country_population": population,
        "country_capital": capital,
        "country_currency": currency,
        "country_language": language,
        "country_summary": country_summary,
    }

In [14]:
def fallback_country_info(country_name: str) -> dict[str, Any]:
    return {
        "country_region": "unknown",
        "country_subregion": "unknown",
        "country_population": 0,
        "country_capital": "unknown",
        "country_currency": "unknown",
        "country_language": "unknown",
        "country_summary": f"Country metadata is unavailable for {country_name}.",
    }

In [15]:
def build_travel_profile(city_name: str) -> dict[str, Any]:
    print("Building profile for:", city_name)

    geocoding_data = fetch_geocoding(city_name)
    location = normalize_geocoding(city_name, geocoding_data)

    weather_data = fetch_weather_forecast(
        latitude=location["latitude"],
        longitude=location["longitude"],
        forecast_days=7,
    )

    weather = normalize_weather_forecast(weather_data)

    try:
        country_data = fetch_country_info(location["country"])
        country = normalize_country_info(country_data)
    except Exception as error:
        print("Country API failed:", location["country"], repr(error))
        country = fallback_country_info(location["country"])

    travel_style = infer_travel_style(
        city=location["city"],
        country=location["country"],
        weather_label=weather["weather_label"],
        avg_max_temp=weather["avg_max_temp_c"],
    )

    combined_summary = (
        f"Travel destination profile for {location['city']}, {location['country']}. "
        f"Region: {location['admin1']}. "
        f"Timezone: {location['timezone']}. "
        f"Population: {location['population']}. "
        f"Travel style: {travel_style}. "
        f"{weather['weather_summary']} "
        f"{country['country_summary']}"
    )

    return {
        **location,
        **weather,
        **country,
        "travel_style": travel_style,
        "combined_summary": combined_summary,
    }

In [16]:
def infer_travel_style(
    *,
    city: str,
    country: str,
    weather_label: str,
    avg_max_temp: float,
) -> str:
    coastal_cities = {
        "Barcelona",
        "Lisbon",
        "Dubrovnik",
        "Malaga",
        "Nice",
    }

    historic_cities = {
        "Rome",
        "Athens",
        "Prague",
    }

    styles = []

    if city in coastal_cities:
        styles.append("coast")
        styles.append("sea views")

    if city in historic_cities:
        styles.append("history")
        styles.append("architecture")

    if avg_max_temp >= 20:
        styles.append("warm weather")

    if weather_label in {"pleasant", "mild"}:
        styles.append("walking")

    styles.append("city break")
    styles.append("food")

    return ", ".join(dict.fromkeys(styles))

In [17]:
travel_records = []

for city_name in destination_names:
    try:
        record = build_travel_profile(city_name)
        travel_records.append(record)
    except Exception as error:
        print("Failed:", city_name)
        print(repr(error))

print("Travel records:", len(travel_records))

Building profile for: Barcelona
Building profile for: Rome
Building profile for: Athens
Building profile for: Lisbon
Building profile for: Dubrovnik
Building profile for: Prague
Building profile for: Malaga
Building profile for: Nice
Travel records: 8


In [18]:
for record in travel_records:
    print(record["city"], record["country"])
    print("Travel style:", record["travel_style"])
    print("Weather:", record["weather_label"])
    print("Avg max temp:", record["avg_max_temp_c"])
    print("Rain probability:", record["avg_rain_probability"])
    print("Currency:", record["country_currency"])
    print("Summary:", record["combined_summary"][:700])
    print("-" * 100)

Barcelona Spain
Travel style: coast, sea views, warm weather, walking, city break, food
Weather: pleasant
Avg max temp: 24.73
Rain probability: 18.57
Currency: EUR
Summary: Travel destination profile for Barcelona, Spain. Region: Catalonia. Timezone: Europe/Madrid. Population: 1686208. Travel style: coast, sea views, warm weather, walking, city break, food. Average max temperature: 24.7°C. Average min temperature: 19.2°C. Average rain probability: 18.6%. Maximum wind speed: 17.6 km/h. Typical weather: overcast. Weather label: pleasant. Spain is located in Europe, Southern Europe. Capital: Madrid. Population: 49315949. Currency: EUR. Main language: Spanish.
----------------------------------------------------------------------------------------------------
Rome Italy
Travel style: history, architecture, warm weather, city break, food
Weather: hot
Avg max temp: 29.77
Rain probability: 1.71
Currency: EUR
Summary: Travel destination profile for Rome, Italy. Region: Lazio. Timezone: Europe/

In [19]:
COLLECTION_NAME = "GoogleTravelWeatherProfile"

if client.collections.exists(COLLECTION_NAME):
    client.collections.delete(COLLECTION_NAME)

travel_profiles = client.collections.create(
    name=COLLECTION_NAME,
    vector_config=wvc.config.Configure.Vectors.text2vec_google_gemini(
        name="travel_vector",
        source_properties=[
            "city",
            "country",
            "travel_style",
            "weather_summary",
            "country_summary",
            "combined_summary",
        ],
        model=GOOGLE_EMBEDDING_MODEL,
    ),
    generative_config=wvc.config.Configure.Generative.google_gemini(
        model=GOOGLE_GENERATIVE_MODEL,
        temperature=0.2,
        max_output_tokens=700,
    ),
    properties=[
        wvc.config.Property(name="city", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="country", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="country_code", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="admin1", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="latitude", data_type=wvc.config.DataType.NUMBER),
        wvc.config.Property(name="longitude", data_type=wvc.config.DataType.NUMBER),
        wvc.config.Property(name="timezone", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="population", data_type=wvc.config.DataType.NUMBER),
        wvc.config.Property(name="avg_max_temp_c", data_type=wvc.config.DataType.NUMBER),
        wvc.config.Property(name="avg_min_temp_c", data_type=wvc.config.DataType.NUMBER),
        wvc.config.Property(name="avg_rain_probability", data_type=wvc.config.DataType.NUMBER),
        wvc.config.Property(name="max_wind_speed_kmh", data_type=wvc.config.DataType.NUMBER),
        wvc.config.Property(name="weather_description", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="weather_label", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="weather_summary", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="country_region", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="country_subregion", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="country_population", data_type=wvc.config.DataType.NUMBER),
        wvc.config.Property(name="country_capital", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="country_currency", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="country_language", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="country_summary", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="travel_style", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="combined_summary", data_type=wvc.config.DataType.TEXT),
    ],
)

print("Created collection:", COLLECTION_NAME)

Created collection: GoogleTravelWeatherProfile


In [20]:
if not travel_records:
    raise RuntimeError("No travel records to import.")

result = travel_profiles.data.insert_many(travel_records)

if result.errors:
    print("Import errors:")
    print(result.errors)
else:
    print("Import completed.")

Import completed.


In [21]:
response = travel_profiles.aggregate.over_all(total_count=True)

print("Total objects:", response.total_count)

Total objects: 8


In [22]:
response = travel_profiles.query.fetch_objects(
    limit=10,
    return_properties=[
        "city",
        "country",
        "travel_style",
        "weather_label",
        "avg_max_temp_c",
        "avg_rain_probability",
        "country_currency",
        "country_language",
    ],
)

for obj in response.objects:
    print("UUID:", obj.uuid)
    print("City:", obj.properties["city"])
    print("Country:", obj.properties["country"])
    print("Style:", obj.properties["travel_style"])
    print("Weather:", obj.properties["weather_label"])
    print("Avg max temp:", obj.properties["avg_max_temp_c"])
    print("Rain probability:", obj.properties["avg_rain_probability"])
    print("Currency:", obj.properties["country_currency"])
    print("Language:", obj.properties["country_language"])
    print("-" * 100)

UUID: 0be4b6a2-3724-41f8-8739-e80e62c684e1
City: Athens
Country: Greece
Style: history, architecture, warm weather, city break, food
Weather: hot
Avg max temp: 31.46
Rain probability: 1.29
Currency: EUR
Language: Greek
----------------------------------------------------------------------------------------------------
UUID: 45d4e68b-bd7a-486b-8ffc-cb6cd8ccc952
City: Dubrovnik
Country: Croatia
Style: coast, sea views, warm weather, walking, city break, food
Weather: pleasant
Avg max temp: 27.36
Rain probability: 4.71
Currency: EUR
Language: Croatian
----------------------------------------------------------------------------------------------------
UUID: 4b5589f4-134d-444e-b7b0-dfbee2cfa533
City: Barcelona
Country: Spain
Style: coast, sea views, warm weather, walking, city break, food
Weather: pleasant
Avg max temp: 24.73
Rain probability: 18.57
Currency: EUR
Language: Spanish
----------------------------------------------------------------------------------------------------
UUID: 5b00

In [23]:
response = travel_profiles.query.near_text(
    query="best city break with architecture, walking and good food",
    target_vector="travel_vector",
    limit=5,
    return_properties=[
        "city",
        "country",
        "travel_style",
        "weather_label",
        "avg_max_temp_c",
        "avg_rain_probability",
        "combined_summary",
    ],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    print("City:", obj.properties["city"])
    print("Country:", obj.properties["country"])
    print("Style:", obj.properties["travel_style"])
    print("Weather:", obj.properties["weather_label"])
    print("Avg max temp:", obj.properties["avg_max_temp_c"])
    print("Rain probability:", obj.properties["avg_rain_probability"])
    print("-" * 100)

Distance: 0.3810896873474121
City: Lisbon
Country: Portugal
Style: coast, sea views, warm weather, walking, city break, food
Weather: pleasant
Avg max temp: 24.73
Rain probability: 0.0
----------------------------------------------------------------------------------------------------
Distance: 0.38128453493118286
City: Prague
Country: Czechia
Style: history, architecture, warm weather, walking, city break, food
Weather: pleasant
Avg max temp: 22.69
Rain probability: 38.86
----------------------------------------------------------------------------------------------------
Distance: 0.38625526428222656
City: Barcelona
Country: Spain
Style: coast, sea views, warm weather, walking, city break, food
Weather: pleasant
Avg max temp: 24.73
Rain probability: 18.57
----------------------------------------------------------------------------------------------------
Distance: 0.3918675184249878
City: Rome
Country: Italy
Style: history, architecture, warm weather, city break, food
Weather: hot
Avg

In [24]:
filters = (
    Filter.by_property("avg_rain_probability").less_or_equal(50)
    & Filter.by_property("avg_max_temp_c").greater_or_equal(12)
)

response = travel_profiles.query.near_text(
    query="warm destination for sightseeing, walking and sea views",
    target_vector="travel_vector",
    filters=filters,
    limit=5,
    return_properties=[
        "city",
        "country",
        "travel_style",
        "weather_label",
        "avg_max_temp_c",
        "avg_rain_probability",
        "max_wind_speed_kmh",
        "combined_summary",
    ],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    print("City:", obj.properties["city"])
    print("Country:", obj.properties["country"])
    print("Style:", obj.properties["travel_style"])
    print("Weather:", obj.properties["weather_label"])
    print("Avg max temp:", obj.properties["avg_max_temp_c"])
    print("Rain probability:", obj.properties["avg_rain_probability"])
    print("Wind:", obj.properties["max_wind_speed_kmh"])
    print("-" * 100)

Distance: 0.3071105480194092
City: Lisbon
Country: Portugal
Style: coast, sea views, warm weather, walking, city break, food
Weather: pleasant
Avg max temp: 24.73
Rain probability: 0.0
Wind: 24.4
----------------------------------------------------------------------------------------------------
Distance: 0.3166460394859314
City: Málaga
Country: Spain
Style: warm weather, walking, city break, food
Weather: pleasant
Avg max temp: 26.54
Rain probability: 0.86
Wind: 25.0
----------------------------------------------------------------------------------------------------
Distance: 0.3183300495147705
City: Dubrovnik
Country: Croatia
Style: coast, sea views, warm weather, walking, city break, food
Weather: pleasant
Avg max temp: 27.36
Rain probability: 4.71
Wind: 13.3
----------------------------------------------------------------------------------------------------
Distance: 0.31884515285491943
City: Nice
Country: France
Style: coast, sea views, warm weather, walking, city break, food
Weat

In [25]:
response = travel_profiles.query.fetch_objects(
    filters=Filter.by_property("country_currency").equal("EUR"),
    limit=20,
    return_properties=[
        "city",
        "country",
        "country_currency",
        "country_language",
        "travel_style",
        "weather_label",
        "avg_max_temp_c",
    ],
)

for obj in response.objects:
    print("City:", obj.properties["city"])
    print("Country:", obj.properties["country"])
    print("Currency:", obj.properties["country_currency"])
    print("Language:", obj.properties["country_language"])
    print("Style:", obj.properties["travel_style"])
    print("Weather:", obj.properties["weather_label"])
    print("-" * 100)

City: Rome
Country: Italy
Currency: EUR
Language: Italian
Style: history, architecture, warm weather, city break, food
Weather: hot
----------------------------------------------------------------------------------------------------
City: Barcelona
Country: Spain
Currency: EUR
Language: Spanish
Style: coast, sea views, warm weather, walking, city break, food
Weather: pleasant
----------------------------------------------------------------------------------------------------
City: Athens
Country: Greece
Currency: EUR
Language: Greek
Style: history, architecture, warm weather, city break, food
Weather: hot
----------------------------------------------------------------------------------------------------
City: Lisbon
Country: Portugal
Currency: EUR
Language: Portuguese
Style: coast, sea views, warm weather, walking, city break, food
Weather: pleasant
----------------------------------------------------------------------------------------------------
City: Dubrovnik
Country: Croatia
Cur

In [26]:
response = travel_profiles.generate.near_text(
    query="recommend a short holiday destination for architecture, food, walking and pleasant weather",
    target_vector="travel_vector",
    grouped_task=(
        "Act as a travel planning assistant. "
        "Use only the retrieved travel destination profiles. "
        "Recommend the best destination for a short holiday. "
        "Mention city, country, travel style, weather, rain probability, currency and language. "
        "Do not invent facts outside the retrieved records."
    ),
    limit=5,
    return_properties=[
        "city",
        "country",
        "travel_style",
        "weather_label",
        "avg_max_temp_c",
        "avg_rain_probability",
        "country_currency",
        "country_language",
        "combined_summary",
    ],
)

print(response.generated)

print("\nSources:")
for obj in response.objects:
    print(
        "-",
        obj.properties["city"],
        obj.properties["country"],
        "| weather:",
        obj.properties["weather_label"],
        "| temp:",
        obj.properties["avg_max_temp_c"],
        "| rain:",
        obj.properties["avg_rain_probability"],
    )

For a short holiday, **Lisbon, Portugal** is an excellent choice.

Here are the details:
*   **City:** Lisbon
*   **Country:** Portugal
*   **Travel Style:** Coast, sea views, warm weather, walking, city break, food
*   **Weather:** Pleasant, with an average maximum temperature of 24.7°C and an average minimum temperature of 16.0°C.
*   **Rain Probability:** 0.0%
*   **Currency:** EUR
*   **Language:** Portuguese

Sources:
- Prague Czechia | weather: pleasant | temp: 22.69 | rain: 38.86
- Lisbon Portugal | weather: pleasant | temp: 24.73 | rain: 0.0
- Barcelona Spain | weather: pleasant | temp: 24.73 | rain: 18.57
- Málaga Spain | weather: pleasant | temp: 26.54 | rain: 0.86
- Nice France | weather: pleasant | temp: 24.04 | rain: 9.29


In [27]:
def recommend_destination(
    user_need: str,
    *,
    max_rain_probability: float | None = None,
    min_avg_max_temp: float | None = None,
    country: str | None = None,
    limit: int = 5,
) -> None:
    filters = None

    if max_rain_probability is not None:
        rain_filter = Filter.by_property("avg_rain_probability").less_or_equal(
            max_rain_probability
        )
        filters = rain_filter if filters is None else filters & rain_filter

    if min_avg_max_temp is not None:
        temp_filter = Filter.by_property("avg_max_temp_c").greater_or_equal(
            min_avg_max_temp
        )
        filters = temp_filter if filters is None else filters & temp_filter

    if country is not None:
        country_filter = Filter.by_property("country").equal(country)
        filters = country_filter if filters is None else filters & country_filter

    response = travel_profiles.generate.near_text(
        query=user_need,
        target_vector="travel_vector",
        filters=filters,
        grouped_task=(
            "Use only the retrieved travel destination profiles. "
            "Recommend the most suitable destination for the user's need. "
            "Explain the reason using travel style, weather and country metadata. "
            "Do not invent missing facts."
        ),
        limit=limit,
        return_properties=[
            "city",
            "country",
            "travel_style",
            "weather_label",
            "avg_max_temp_c",
            "avg_rain_probability",
            "max_wind_speed_kmh",
            "country_currency",
            "country_language",
            "combined_summary",
        ],
    )

    print("USER NEED:")
    print(user_need)
    print("MAX RAIN:", max_rain_probability)
    print("MIN TEMP:", min_avg_max_temp)
    print("COUNTRY:", country)
    print("-" * 100)

    print(response.generated)

    print("\nRecords used:")
    for obj in response.objects:
        print(
            "-",
            obj.properties["city"],
            obj.properties["country"],
            "| style:",
            obj.properties["travel_style"],
            "| weather:",
            obj.properties["weather_label"],
            "| temp:",
            obj.properties["avg_max_temp_c"],
            "| rain:",
            obj.properties["avg_rain_probability"],
        )

In [28]:
recommend_destination(
    "I want a short city break with architecture, food and walking",
    max_rain_probability=60,
    min_avg_max_temp=10,
)

USER NEED:
I want a short city break with architecture, food and walking
MAX RAIN: 60
MIN TEMP: 10
COUNTRY: None
----------------------------------------------------------------------------------------------------
The user's specific travel needs are not provided in the prompt. Therefore, I cannot recommend the "most suitable" destination as suitability depends entirely on individual preferences (e.g., desired travel style, preferred weather conditions, tolerance for rain, specific interests like history or coastal activities).

Without knowing the user's preferences, it is impossible to determine which of the retrieved profiles (Lisbon, Prague, Barcelona, Rome, Málaga) would be the most suitable. I cannot invent missing facts about the user's needs.

Please provide the user's travel needs (e.g., "looking for a historical trip with warm weather and low rain," or "want a city break with sea views and good food") to receive a tailored recommendation.

Records used:
- Lisbon Portugal | st

In [29]:
recommend_destination(
    "I want a warm coastal destination with sea views and relaxed sightseeing",
    max_rain_probability=60,
    min_avg_max_temp=15,
)

USER NEED:
I want a warm coastal destination with sea views and relaxed sightseeing
MAX RAIN: 60
MIN TEMP: 15
COUNTRY: None
----------------------------------------------------------------------------------------------------
Please provide the user's travel needs so I can recommend the most suitable destination. Without knowing what the user is looking for (e.g., specific activities, preferred temperature range, desire for a quiet or lively place, budget, etc.), I cannot make an informed recommendation from the provided profiles.

Records used:
- Nice France | style: coast, sea views, warm weather, walking, city break, food | weather: pleasant | temp: 24.04 | rain: 9.29
- Lisbon Portugal | style: coast, sea views, warm weather, walking, city break, food | weather: pleasant | temp: 24.73 | rain: 0.0
- Barcelona Spain | style: coast, sea views, warm weather, walking, city break, food | weather: pleasant | temp: 24.73 | rain: 18.57
- Dubrovnik Croatia | style: coast, sea views, warm weath

In [30]:
recommend_destination(
    "I want a historical city with architecture and mild weather",
    max_rain_probability=60,
    min_avg_max_temp=10,
)

USER NEED:
I want a historical city with architecture and mild weather
MAX RAIN: 60
MIN TEMP: 10
COUNTRY: None
----------------------------------------------------------------------------------------------------
Based on the provided travel destination profiles, **Lisbon, Portugal** is the most suitable destination.

Here's why:

*   **Travel Style**: Lisbon offers a diverse and appealing travel style

Records used:
- Lisbon Portugal | style: coast, sea views, warm weather, walking, city break, food | weather: pleasant | temp: 24.73 | rain: 0.0
- Prague Czechia | style: history, architecture, warm weather, walking, city break, food | weather: pleasant | temp: 22.69 | rain: 38.86
- Barcelona Spain | style: coast, sea views, warm weather, walking, city break, food | weather: pleasant | temp: 24.73 | rain: 18.57
- Athens Greece | style: history, architecture, warm weather, city break, food | weather: hot | temp: 31.46 | rain: 1.29
- Málaga Spain | style: warm weather, walking, city break,

In [31]:
client.close()

print("Client closed.")

Client closed.
